## 스스로 해보자_코드 기록
### D006 - 대용량 데이터는 어떻게 처리할까 ? - pandas의 한계, 청크, 그리고 Polars
---

In [ ]:
# 스스로 해보자! (dtype 최적화) ✏️
# 1. orders의 현재 메모리를 측정해서 출력하세요.
# 2. quantity, channel, customer_id, product_id의 dtype을 줄여 'orders_small'을 만드세요.
# 3. 줄인 뒤 메모리와 절감률을 출력하세요.

# 바이트 값 / 1024        → KB로 변환
# 바이트 값 / 1024 / 1024 → MB로 변환   (= /1024**2)
print((orders.memory_usage(deep=True).sum() / 1024 / 1024).round(2) )

orders_small = orders.copy()
orders_small["quantity"] = orders["quantity"].astype("int32")
orders_small["channel"] = orders["channel"].astype("category")
orders_small["customer_id"] = orders["customer_id"].astype("category")
orders_small["product_id"] = orders["product_id"].astype("category")

b = orders.memory_usage(deep=True).sum() / 1024 / 1024
a = orders_small.memory_usage(deep=True).sum() / 1024 / 1024
print(f"Before: {b:.2f} MB")
print(f"After : {a:.2f} MB")
print(f"절감   : {(1 - a / b):.2f}")

In [ ]:
# 스스로 해보자! (polars 입문) ✏️
# 1. logs_pl을 group_by('device')로 묶으세요.
# 2. bytes_sent의 합계를 구하세요.
# 3. 합계 컬럼명을 'total_bytes'로 바꾸세요.

ans = (
    logs_pl
    .group_by("device")
    .agg(pl.col("bytes_sent").sum().alias("total_bytes"))
)

print(ans)

In [ ]:
# 스스로 해보자! (Lazt evaluation) ✏️
# 1. pl.scan_csv로 logs_csv를 lazy 모드로 엽니다.
# 2. status_code >= 400 인 행만 필터링하세요.
# 3. page별로 row 개수를 세고, 많은 순으로 정렬하세요.
# 4. .collect()로 실행하세요.

error_summary = (
    pl.scan_csv(logs_csv)
    .filter(pl.col("status_code") >= 400)
    .group_by("page")
    .agg(pl.len().alias("n_errors"))
    .collect()
)
print(error_summary)

In [ ]:
# 스스로 해보자! (pandas vs Polars) ✏️
# 1. 본인이 보고 싶은 쿼리를 정합니다. (예: 채널이 'mobile'이고 status==200인 행의 page별 평균 bytes_sent)
# 2. 같은 쿼리를 pandas / polars eager / polars lazy 로 작성하세요.
# 3. benchmark()로 세 가지 시간을 출력하세요.

print(pd.read_csv(logs_csv, nrows=5).columns.tolist())

# channel='mobile' & status_code==200 인 행의 page별 평균 bytes_sent

def q5_pandas():
    df = pd.read_csv(logs_csv)
    filtered = df[(df["device"] == "mobile") & (df["status_code"] == 200)]
    return filtered.groupby("page")["bytes_sent"].mean()

def q5_polars_eager():
    return (
        pl.read_csv(logs_csv)
        .filter((pl.col("device") == "mobile") & (pl.col("status_code") == 200))
        .group_by("page")
        .agg(pl.col("bytes_sent").mean())
    )

def q5_polars_lazy():
    return (
        pl.scan_csv(logs_csv)
        .filter((pl.col("device") == "mobile") & (pl.col("status_code") == 200))
        .group_by("page")
        .agg(pl.col("bytes_sent").mean())
        .collect()
    )

print("=== Q5: device=mobile & status=200 → page별 평균 bytes_sent ===")
q5_pd = benchmark("pandas",        q5_pandas)
q5_pe = benchmark("polars (eager)", q5_polars_eager)
q5_pl = benchmark("polars (lazy)",  q5_polars_lazy)

In [ ]:
# 스스로 해보자! (도구 선택 의사결정) ✏️
# 위 시나리오 A, B, C에 대해 본인의 선택을 주석으로 적어보세요.
# 예) A: pandas — 작고 일회성이라 새 도구 도입 비용 무의미

# A: pandas : 데이터 양이 적으며, 매주 돌려야 하는 만큼 빠르게 결과만 볼 수 있어야 하기 때문
# B: Polars (lazy) : B는 데이터 양이 크고 ETL인 만큼 원본 데이터 크기 보다 더 많은 메모리를 필요로 함
# C: pandas + dtype 최적화 : 일단 팀 내부에서 pandas를 사용함, 그러나 데이터 크기가 작진 않기 때문에 최적화 진행